# Silver layer tranformation and data cleansing of Customer Product data from crm volume
Location: _workspace.bronze.crm_prd_info_

1. Check data types
2. Rename column headers
3. Normalize categorical data

### Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

## Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
df.display()

print(df.dtypes)

In [0]:
# Trimming

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Product key parsing and creating category_id column

In [0]:

df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

df.display()

## Normalization of Product Line column

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)
df.display()

## Fix possible missing values in Product cost:

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

## Rename columns

In [0]:
rename_map = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date",
    "cat_id": "category_id"
    }

for old, new in rename_map.items():
    df = df.withColumnRenamed(old, new)

df.display()

## Write into Silver layer in the catalog

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_product")